##### Setup

###### 1.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

###### 2.

In [3]:
def exibir_controles(funcao, controles):
    # interactive_output executa com os defaults e observa alterações posteriores.
    saida = widgets.interactive_output(funcao, controles)
    display(widgets.VBox(list(controles.values())), saida)
    return saida


def finalizar_figura(figura, arquivo):
    figura.tight_layout()
    figura.savefig(arquivo, dpi=140, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(figura)


def deslizador(valor, minimo, maximo, passo, descricao):
    return widgets.FloatSlider(value=valor, min=minimo, max=maximo, step=passo,
                               description=descricao, continuous_update=False)

###### 3.

In [4]:
EPS = 1e-9


def produto_vetorial_2d(a, b):
    return a[0] * b[1] - a[1] * b[0]


def validar_triangulos(triangulos):
    for i, vertices in enumerate(triangulos, 1):
        if not np.all(np.isfinite(vertices)):
            raise ValueError(f"Triângulo {i}: coordenadas não finitas")
        if abs(produto_vetorial_2d(vertices[1]-vertices[0], vertices[2]-vertices[0])) < EPS:
            raise ValueError(f"Triângulo {i}: os vértices são colineares ou coincidentes")


def ponto_no_triangulo(ponto, vertices):
    sinais = [produto_vetorial_2d(vertices[(i+1)%3]-vertices[i], ponto-vertices[i])
              for i in range(3)]
    return min(sinais) >= -EPS or max(sinais) <= EPS


def intersecao_triangulo(origem, destino, vertices):
    """Primeiro encontro no segmento fechado, incluindo bordas e colinearidade."""
    direcao = np.asarray(destino, float) - origem
    norma2 = np.dot(direcao, direcao)
    if norma2 < EPS**2:
        return None
    candidatos = []
    if ponto_no_triangulo(origem, vertices):
        candidatos.append((0.0, None))
    for i in range(3):
        a, b = vertices[i], vertices[(i+1)%3]
        aresta = b-a
        denominador = produto_vetorial_2d(direcao, aresta)
        deslocamento = a-origem
        if abs(denominador) > EPS:
            t = produto_vetorial_2d(deslocamento, aresta)/denominador
            u = produto_vetorial_2d(deslocamento, direcao)/denominador
            if -EPS <= t <= 1+EPS and -EPS <= u <= 1+EPS:
                candidatos.append((float(np.clip(t, 0, 1)), i))
        elif abs(produto_vetorial_2d(deslocamento, direcao)) <= EPS:
            extremos = sorted((np.dot(a-origem, direcao)/norma2,
                                np.dot(b-origem, direcao)/norma2))
            inicio, fim = max(0, extremos[0]), min(1, extremos[1])
            if inicio <= fim + EPS:
                candidatos.append((float(np.clip(inicio, 0, 1)), i))
    if not candidatos:
        return None
    t, aresta = min(candidatos, key=lambda item: item[0])
    return {'t': t, 'ponto': origem+t*direcao, 'aresta': aresta,
            'distancia': t*np.sqrt(norma2)}


def encontros_cena(origem, destino, triangulos):
    encontros = []
    for i, vertices in enumerate(triangulos):
        encontro = intersecao_triangulo(origem, destino, vertices)
        if encontro is not None:
            encontros.append(dict(encontro, triangulo=i))
    return sorted(encontros, key=lambda item: (item['t'], item['triangulo']))




def desenhar_cena(eixo, triangulos, ca, p, titulo):
    cores = plt.get_cmap('tab10').colors
    for i, vertices in enumerate(triangulos):
        eixo.fill(*vertices.T, color=cores[i%10], alpha=0.35,
                  edgecolor=cores[i%10], label=f'Triângulo {i+1}')
        eixo.text(*vertices.mean(axis=0), f'T{i+1}', ha='center')
    eixo.plot([ca[0], p[0]], [ca[1], p[1]], 'k--', label='Segmento CA–P')
    eixo.scatter(*ca, color='black', zorder=5, label='CA')
    eixo.scatter(*p, color='red', zorder=5, label='P')
    eixo.set(title=titulo, xlabel='x', ylabel='y', aspect='equal')
    eixo.grid(alpha=0.25)
    eixo.legend(fontsize=8, loc='best')


TRIANGULOS_BASE = np.array([[[1, 3], [2, 5], [3, 3]],
                            [[2, 2], [3, 4], [4, 2]]], dtype=float)

##### QUESTÃO1

###### 1.

In [5]:
# Defaults preservam as coordenadas do PDF; os controles editam todos os vértices.
controles_q1 = {
    'ca_x': deslizador(0, -5, 8, 0.1, 'CA x'),
    'ca_y': deslizador(0, -5, 8, 0.1, 'CA y'),
    'p_x': deslizador(3, -5, 8, 0.1, 'P x'),
    'p_y': deslizador(3, -5, 8, 0.1, 'P y'),
}
for i, vertices in enumerate(TRIANGULOS_BASE):
    for j, ponto in enumerate(vertices):
        for k, coordenada in enumerate('xy'):
            controles_q1[f't{i}_{j}_{coordenada}'] = deslizador(
                ponto[k], -5, 8, 0.1, f'T{i+1} V{j+1} {coordenada}')


def ler_cena_q1(valores):
    triangulos = np.array([[[valores[f't{i}_{j}_x'], valores[f't{i}_{j}_y']]
                           for j in range(3)] for i in range(2)])
    return triangulos, np.array([valores['ca_x'], valores['ca_y']]), np.array([valores['p_x'], valores['p_y']])

###### 2.

In [6]:
def mostrar_cena_q1(**valores):
    triangulos, ca, p = ler_cena_q1(valores)
    figura, eixo = plt.subplots(figsize=(7, 6))
    desenhar_cena(eixo, triangulos, ca, p, 'Cena 2D — Modelagem e projeção')
    finalizar_figura(figura, 'lista5_questao1_cena.png')


saida_cena_q1 = exibir_controles(mostrar_cena_q1, controles_q1)

Output()

###### 3.

In [7]:
def mostrar_visibilidade_q1(**valores):
    triangulos, ca, p = ler_cena_q1(valores)
    try:
        validar_triangulos(triangulos)
    except ValueError as erro:
        print(erro)
        return
    encontros = encontros_cena(ca, p, triangulos)
    figura, eixo = plt.subplots(figsize=(7, 6))
    if np.linalg.norm(p-ca) < EPS:
        titulo = 'CA e P coincidem: segmento sem direção'
    elif encontros:
        primeiros = [e for e in encontros if abs(e['t']-encontros[0]['t']) < EPS]
        titulo = ('Triângulo visível: ' if len(primeiros) == 1 else 'Empate no primeiro encontro: ') + ', '.join(f"T{e['triangulo']+1}" for e in primeiros)
        for encontro in encontros:
            eixo.scatter(*encontro['ponto'], color='magenta', marker='x', s=90, zorder=6)
            print(f"T{encontro['triangulo']+1}: ponto {np.round(encontro['ponto'], 3)}, distância {encontro['distancia']:.3f}")
        eixo.plot(*np.vstack((ca, encontros[0]['ponto'])).T, color='green', linewidth=3)
    else:
        titulo = 'Nenhum triângulo intersectado'
    print(titulo)
    desenhar_cena(eixo, triangulos, ca, p, titulo)
    finalizar_figura(figura, 'lista5_questao1_visibilidade.png')


saida_visibilidade_q1 = exibir_controles(mostrar_visibilidade_q1, controles_q1)

Output()

##### QUESTÃO2

###### 1.

In [8]:
def normal_encontro(encontro, triangulos, direcao):
    i = encontro['aresta']
    if i is None:
        return -direcao / np.linalg.norm(direcao)
    vertices = triangulos[encontro['triangulo']]
    aresta = vertices[(i+1)%3]-vertices[i]
    normal = np.array([-aresta[1], aresta[0]]) / np.linalg.norm(aresta)
    return normal if np.dot(normal, direcao) <= 0 else -normal


# O terceiro triângulo bloqueia parte da luz na configuração inicial.
triangulos_extra = np.concatenate((TRIANGULOS_BASE.copy(), np.array([
    [[-1, 3], [-1, 5], [1, 5]],
    [[4, 4], [5, 6], [6, 4]],
    [[0, -2], [2, -1], [3, -2]],
    [[5, 0], [6, 2], [7, 0]],
], dtype=float)))
quantidade_q2 = widgets.IntSlider(value=3, min=3, max=6, description='Triângulos', continuous_update=False)
selecao_q2 = widgets.Dropdown(options=[('T1', 0), ('T2', 1), ('T3', 2)], value=0, description='Editar')
editores_q2 = {f'v{j}_{c}': deslizador(triangulos_extra[0, j, k], -6, 9, 0.1, f'V{j+1} {c}')
               for j in range(3) for k, c in enumerate('xy')}
controles_q2 = {
    'quantidade': quantidade_q2,
    'ca_x': deslizador(0, -6, 9, 0.1, 'CA x'),
    'ca_y': deslizador(0, -6, 9, 0.1, 'CA y'),
    'p_x': deslizador(3, -6, 9, 0.1, 'P x'),
    'p_y': deslizador(3, -6, 9, 0.1, 'P y'),
    'luz_x': deslizador(-2, -6, 9, 0.1, 'Luz x'),
    'luz_y': deslizador(6, -6, 9, 0.1, 'Luz y'),
    **editores_q2,
}
carregando_vertices_q2 = False


def carregar_editores_q2(mudanca):
    global carregando_vertices_q2
    carregando_vertices_q2 = True
    for j in range(3):
        for k, c in enumerate('xy'):
            editores_q2[f'v{j}_{c}'].value = triangulos_extra[selecao_q2.value, j, k]
    carregando_vertices_q2 = False
    # Força atualização após carregar o conjunto completo de coordenadas.
    editores_q2['v0_x'].notify_change({'name': 'value', 'type': 'change',
        'old': editores_q2['v0_x'].value, 'new': editores_q2['v0_x'].value, 'owner': editores_q2['v0_x']})


def atualizar_quantidade_q2(mudanca):
    selecao_q2.options = [(f'T{i+1}', i) for i in range(mudanca['new'])]


quantidade_q2.observe(atualizar_quantidade_q2, names='value')
selecao_q2.observe(carregar_editores_q2, names='value')
display(selecao_q2)


def ler_cena_q2(valores):
    if carregando_vertices_q2:
        return None
    triangulos_extra[selecao_q2.value] = [[valores[f'v{j}_x'], valores[f'v{j}_y']] for j in range(3)]
    triangulos = triangulos_extra[:valores['quantidade']].copy()
    validar_triangulos(triangulos)
    ca = np.array([valores['ca_x'], valores['ca_y']])
    p = np.array([valores['p_x'], valores['p_y']])
    luz = np.array([valores['luz_x'], valores['luz_y']])
    return triangulos, ca, p, luz


def mostrar_modelagem_q2(**valores):
    try:
        cena = ler_cena_q2(valores)
        if cena is None:
            return
        triangulos, ca, p, luz = cena
    except ValueError as erro:
        print(erro)
        return
    figura, eixo = plt.subplots(figsize=(8, 6))
    desenhar_cena(eixo, triangulos, ca, p, 'Cena ampliada')
    eixo.scatter(*luz, marker='*', color='gold', s=180, edgecolors='black', label='Luz')
    eixo.legend(fontsize=8)
    finalizar_figura(figura, 'lista5_questao2_cena.png')


saida_modelagem_q2 = exibir_controles(mostrar_modelagem_q2, controles_q2)

Dropdown(description='Editar', options=(('T1', 0), ('T2', 1), ('T3', 2)), value=0)

Output()

###### 2.

In [9]:
def iluminacao_direta(encontro, normal, luz, triangulos):
    ponto = encontro['ponto']
    vetor = luz-ponto
    distancia = np.linalg.norm(vetor)
    if distancia < EPS:
        return 0.0, False
    direcao = vetor/distancia
    cosseno = max(float(np.dot(normal, direcao)), 0.0)
    # O deslocamento e o limite antes da luz eliminam o contato com a própria face.
    origem = ponto + normal*1e-6
    destino = luz-direcao*1e-6
    bloqueado = bool(encontros_cena(origem, destino, triangulos))
    intensidade = 0.0 if bloqueado else cosseno/(1+0.05*distancia**2)
    return intensidade, bloqueado


def mostrar_sombras_q2(**valores):
    try:
        cena = ler_cena_q2(valores)
        if cena is None:
            return
        triangulos, ca, p, luz = cena
    except ValueError as erro:
        print(erro)
        return
    encontros = encontros_cena(ca, p, triangulos)
    figura, eixo = plt.subplots(figsize=(8, 6))
    titulo = 'Nenhum encontro no segmento CA–P'
    if encontros:
        encontro = encontros[0]
        normal = normal_encontro(encontro, triangulos, p-ca)
        intensidade, bloqueado = iluminacao_direta(encontro, normal, luz, triangulos)
        titulo = f"T{encontro['triangulo']+1}: {'em sombra' if bloqueado else 'luz sem bloqueio'}"
        eixo.plot(*np.vstack((encontro['ponto'], luz)).T,
                  color='red' if bloqueado else 'gold', linewidth=2, label='Raio de luz')
        eixo.scatter(*encontro['ponto'], color='magenta', zorder=6)
        eixo.quiver(*encontro['ponto'], *normal, angles='xy', scale_units='xy', scale=2)
        print(f'Iluminação direta: {intensidade:.4f}; bloqueio: {bloqueado}')
    desenhar_cena(eixo, triangulos, ca, p, titulo)
    eixo.scatter(*luz, marker='*', s=180, color='gold', edgecolors='black')
    finalizar_figura(figura, 'lista5_questao2_sombras.png')


saida_sombras_q2 = exibir_controles(mostrar_sombras_q2, controles_q2)

Output()

###### 3.

In [10]:
def reflexao_indireta(encontro, normal, triangulos, luz, quantidade, rebotes, refletancia, semente):
    """Estimador didático 2D: direções com distribuição proporcional ao cosseno."""
    rng = np.random.default_rng(semente)
    trajetorias, contribuicoes = [], []
    # A distância cobre a cena; raios que não encontram superfícies escapam.
    pontos = triangulos.reshape(-1, 2)
    alcance = max(10.0, 3*np.linalg.norm(np.ptp(pontos, axis=0)))
    for _ in range(quantidade):
        atual, n = encontro, normal.copy()
        caminho = [atual['ponto'].copy()]
        peso, contribuicao = 1.0, 0.0
        for _ in range(rebotes):
            # Em 2D a densidade normalizada cos(theta)/2 tem esta inversa.
            angulo = np.arcsin(rng.uniform(-1, 1))
            tangente = np.array([-n[1], n[0]])
            direcao = np.cos(angulo)*n + np.sin(angulo)*tangente
            origem = atual['ponto'] + n*1e-6
            encontros = encontros_cena(origem, origem+alcance*direcao, triangulos)
            if not encontros:
                caminho.append(origem + 2*direcao)
                break
            atual = encontros[0]
            caminho.append(atual['ponto'].copy())
            n = normal_encontro(atual, triangulos, direcao)
            direta, _ = iluminacao_direta(atual, n, luz, triangulos)
            peso *= refletancia / (1+0.05*atual['distancia']**2)
            contribuicao += peso*direta
        trajetorias.append(np.array(caminho))
        contribuicoes.append(contribuicao)
    return float(np.mean(contribuicoes)), trajetorias


def mostrar_reflexao_q2(raios, rebotes, refletancia, semente, **valores):
    try:
        cena = ler_cena_q2(valores)
        if cena is None:
            return
        triangulos, ca, p, luz = cena
    except ValueError as erro:
        print(erro)
        return
    encontros = encontros_cena(ca, p, triangulos)
    figura, eixos = plt.subplots(1, 2, figsize=(13, 6))
    if encontros:
        primeiro = encontros[0]
        normal = normal_encontro(primeiro, triangulos, p-ca)
        direta, sombra = iluminacao_direta(primeiro, normal, luz, triangulos)
        indireta, caminhos = reflexao_indireta(primeiro, normal, triangulos, luz,
                                               raios, rebotes, refletancia, semente)
        for caminho in caminhos[:80]:
            eixos[0].plot(*caminho.T, color='purple', alpha=0.25, linewidth=0.8)
        eixos[1].bar(['Direta', 'Direta + indireta'], [direta, direta+indireta], color=['gold', 'purple'])
        eixos[1].set(ylabel='Intensidade estimada', title='Contribuição dos raios secundários', ylim=(0, max(0.1, (direta+indireta)*1.2)))
        print(f"Visível: T{primeiro['triangulo']+1}; sombra: {sombra}; direta={direta:.4f}; indireta={indireta:.4f}")
    else:
        eixos[1].text(0.5, 0.5, 'Nenhum encontro no segmento CA–P', ha='center')
    desenhar_cena(eixos[0], triangulos, ca, p, 'Reflexão difusa — múltiplos raios')
    eixos[0].scatter(*luz, marker='*', s=180, color='gold', edgecolors='black')
    finalizar_figura(figura, 'lista5_questao2_reflexao.png')


controles_reflexao_q2 = dict(controles_q2,
    raios=widgets.IntSlider(value=40, min=5, max=200, step=5, description='Raios', continuous_update=False),
    rebotes=widgets.IntSlider(value=2, min=1, max=5, description='Rebotes', continuous_update=False),
    refletancia=deslizador(0.6, 0, 0.95, 0.05, 'Refletância'),
    semente=widgets.IntSlider(value=42, min=0, max=1000, description='Semente', continuous_update=False))
saida_reflexao_q2 = exibir_controles(mostrar_reflexao_q2, controles_reflexao_q2)

Output()